In [78]:
import pandas as pd

risk_label_map = {
    0: 0, # high risk
    1: 1, # medium risk
    2: 1, # medium risk
    3: 2 # low risk
}

project_name = "stad"

origianal_text_anchor_structured = pd.read_csv("/project/kimlab_tcga/JH_workspace/multimodality_prognosis_prediction/CALM_VLM_refined_anchor/data/diagnostic_description_musk.csv")
original_text_anchor_long_context = pd.read_csv("/project/kimlab_tcga/JH_workspace/multimodality_prognosis_prediction/CALM_VLM_refined_anchor/data/diagnostic_description_bert.csv")
text_reports_structrued = pd.read_csv("/project/kimlab_tcga/JH_workspace/multimodality_prognosis_prediction/CALM_VLM_refined_anchor/data/structured_reports_llama.csv").loc[:, ["patient_id", "structured_report_musk"]]
text_reports_long_context = pd.read_csv("/project/kimlab_tcga/JH_workspace/multimodality_prognosis_prediction/CALM_VLM_refined_anchor/data/long_context_summarized_report_llama.csv").loc[:, ["patient_id", "long_context_summarization"]]
df_meta = pd.read_csv(f"/project/kimlab_tcga/JH_workspace/multimodality_prognosis_prediction/CALM_VLM_refined_anchor/data/OS/tcga_{project_name}_all.csv")
df_meta.label = df_meta.label.map(lambda x: risk_label_map[x])
df_meta

,case_id,slide_id,survival_months,censorship,label
0,TCGA-3M-AB46,TCGA-3M-AB46-01Z-00-DX1.svs,57.98,1,2
1,TCGA-B7-5816,TCGA-B7-5816-01Z-00-DX1.svs,26.68,1,2
2,TCGA-BR-6452,TCGA-BR-6452-01Z-00-DX1.svs,34.66,1,2
3,TCGA-BR-6453,TCGA-BR-6453-01Z-00-DX1.svs,15.93,1,1
4,TCGA-BR-6455,TCGA-BR-6455-01Z-00-DX1.svs,13.86,0,1
...,...,...,...,...,...
344,TCGA-VQ-AA6I,TCGA-VQ-AA6I-01Z-00-DX1.svs,16.13,0,1
345,TCGA-VQ-AA6J,TCGA-VQ-AA6J-01Z-00-DX1.svs,27.53,1,2
346,TCGA-VQ-AA6K,TCGA-VQ-AA6K-01Z-00-DX1.svs,12.42,0,1
347,TCGA-ZA-A8F6,TCGA-ZA-A8F6-01Z-00-DX1.svs,17.25,1,1


In [79]:
sampled_df = df_meta[df_meta.censorship == 0].sample(frac=1.0, random_state=42).groupby("label").head(5).sort_values("label")
# sampled_df = df_meta.sample(frac=1.0, random_state=42).groupby("label").head(50).sort_values("label")
sampled_df = sampled_df.merge(text_reports_structrued, left_on="case_id", right_on="patient_id", how="left").dropna().groupby("label").head(3)
sampled_df = sampled_df.merge(text_reports_long_context, left_on="case_id", right_on="patient_id", how="left").drop(["patient_id_x", "patient_id_y"], axis=1)
sampled_df

,case_id,slide_id,survival_months,censorship,label,structured_report_musk,long_context_summarization
0,TCGA-BR-7901,TCGA-BR-7901-01Z-00-DX1.svs,3.45,0,0,The tumor at the gastroesophageal junction mea...,The cancer is an adenocarcinoma of the gastroe...
1,TCGA-VQ-A94P,TCGA-VQ-A94P-01Z-00-DX1.svs,2.66,0,0,The tumor in the stomach body measures 4.0 cm ...,The cancer is an adenocarcinoma of the stomach...
2,TCGA-BR-A4CS,TCGA-BR-A4CS-01Z-00-DX1.svs,1.48,0,0,The tumor in the stomach fundus measures 5 x 4...,The cancer is an adenocarcinoma of the stomach...
3,TCGA-VQ-A8PD,TCGA-VQ-A8PD-01Z-00-DX1.svs,16.29,0,1,The tumor in the stomach measures 8.7 x 6.5 cm...,The cancer is a moderately differentiated aden...
4,TCGA-D7-6527,TCGA-D7-6527-01Z-00-DX1.svs,10.25,0,1,The tumor in the stomach's cardiac region meas...,The cancer is a papillary adenocarcinoma of th...
5,TCGA-VQ-A91D,TCGA-VQ-A91D-01Z-00-DX1.svs,11.70,0,1,The tumor in the stomach measures 5.0x4.5x4.0 ...,The cancer is an invasive ulcerated moderately...
6,TCGA-BR-8686,TCGA-BR-8686-01Z-00-DX1.svs,20.86,0,2,The tumor in the stomach measures 10 x 8 x 3 c...,The cancer is a poorly differentiated tubular ...
7,TCGA-HJ-7597,TCGA-HJ-7597-01Z-00-DX1.svs,26.45,0,2,The tumor in the stomach has a most significan...,The cancer is an invasive adenocarcinoma of th...
8,TCGA-VQ-A8E3,TCGA-VQ-A8E3-01Z-00-DX1.svs,21.71,0,2,The tumor in the stomach's antrum measures 7.5...,The cancer is a poorly differentiated adenocar...


In [80]:
# for fold in range(5):
#     fold_df = pd.read_csv(f"/project/kimlab_tcga/JH_workspace/multimodality_prognosis_prediction/CALM_VLM_refined_anchor/splits/OS/tcga_{project_name}/splits_{fold}.csv")
    
#     train_ids = fold_df.train[~fold_df.train.isin(sampled_df.case_id.tolist())].tolist()
#     val_ids = fold_df.val[~fold_df.val.isin(sampled_df.case_id.tolist())].dropna().tolist()
    
#     new_fold_df = pd.DataFrame({
#         "train": train_ids,
#         "val": val_ids + [None] * (len(train_ids) - len(val_ids))
#     })
    
#     new_fold_df.to_csv(f"/project/kimlab_tcga/JH_workspace/multimodality_prognosis_prediction/CALM_VLM_refined_anchor/splits/OS/tcga_{project_name}/refined_splits_{fold}.csv", index=False)

In [81]:
low_risk_descriptions = sampled_df[sampled_df.label == 2].structured_report_musk.tolist()
intermediate_risk_descriptions = sampled_df[sampled_df.label == 1].structured_report_musk.tolist()
high_risk_descriptions = sampled_df[sampled_df.label == 0].structured_report_musk.tolist()

low_risk_anchor = origianal_text_anchor_structured.loc[origianal_text_anchor_structured.project == project_name.upper(), "low_risk"].values[0]
intermediate_risk_anchor = origianal_text_anchor_structured.loc[origianal_text_anchor_structured.project == project_name.upper(), "intermediate_risk"].values[0]
high_risk_anchor = origianal_text_anchor_structured.loc[origianal_text_anchor_structured.project == project_name.upper(), "high_risk"].values[0]

In [82]:
print(f"""
You are a clinical pathology writing assistant. Your task is to REWRITE the risk anchors so that each anchor (Low / Intermediate / High) **generalizes across** the provided patient reports while remaining clinically faithful.

### Inputs for {project_name.upper()}
1) Current anchor drafts:
- Low-risk (draft):
{low_risk_anchor}

- Intermediate-risk (draft):
{intermediate_risk_anchor}

- High-risk (draft):
{high_risk_anchor}

2) Patient reports to align with:
- Low-risk examples:
• {low_risk_descriptions[0]}
• {low_risk_descriptions[1]}
• {low_risk_descriptions[2]}

- Intermediate-risk examples:
• {intermediate_risk_descriptions[0]}
• {intermediate_risk_descriptions[1]}
• {intermediate_risk_descriptions[2]}

- High-risk examples:
• {high_risk_descriptions[0]}
• {high_risk_descriptions[1]}
• {high_risk_descriptions[2]}

### What to do
- Start from the current drafts but **refine** them to cover the common patterns across the example reports.
- Prioritize **risk-defining axes** over raw size: differentiation/grade, depth and extent of invasion (organ-wall layers, fat/sinus/peritoneum/adjacent organs), lymph node status (including NX), margin status (especially vascular/soft-tissue margins), and vascular/perineural invasion. Mention size qualitatively only if helpful (e.g., “small/moderate/large”), avoid exact measurements.
- If the disease/site can be inferred from the reports, use **site-appropriate anatomy terms** (e.g., pericolonic fat / renal sinus / perinephric fat). If uncertain, use neutral terms (“adjacent soft tissue”, “regional nodes”).
- **Do not copy** sentences verbatim from the reports. **Abstract and paraphrase**; prefer clinically neutral, guideline-like wording.
- Capture heterogeneity with inclusive phrasing (“confined to … or focal involvement of … without frank invasion of …”; “nodes negative when assessed; NX acceptable”).
- Keep each anchor concise: **2–3 sentences**, **≤ 90 words** per risk.
- **No JSON, no lists, no meta commentary.** Output plain prose only.

### Output format (plain text, exactly this order)
Low-Risk Anchor:
<one short paragraph>

Intermediate-Risk Anchor:
<one short paragraph>

High-Risk Anchor:
<one short paragraph>
""")



You are a clinical pathology writing assistant. Your task is to REWRITE the risk anchors so that each anchor (Low / Intermediate / High) **generalizes across** the provided patient reports while remaining clinically faithful.

### Inputs for STAD
1) Current anchor drafts:
- Low-risk (draft):
The gastric carcinoma is generally confined to the mucosa, submucosa, or superficial muscularis propria without frank penetration through the full gastric wall or direct involvement of adjacent organs. Tumors may be well to poorly differentiated, but regional lymph nodes are negative when assessed, and margins are uninvolved, consistent with complete resection. Vascular or perineural invasion is absent or limited, and residual disease is not identified.

- Intermediate-risk (draft):
The gastric tumor demonstrates deeper invasion through the muscularis propria into perigastric fat or serosal surfaces, but without extensive extension into adjacent organs. Differentiation is usually moderate to poor,

In [83]:
low_risk_descriptions = sampled_df[sampled_df.label == 2].long_context_summarization.tolist()
intermediate_risk_descriptions = sampled_df[sampled_df.label == 1].long_context_summarization.tolist()
high_risk_descriptions = sampled_df[sampled_df.label == 0].long_context_summarization.tolist()

low_risk_anchor = original_text_anchor_long_context.loc[original_text_anchor_long_context.project == project_name.upper(), "low_risk"].values[0]
intermediate_risk_anchor = original_text_anchor_long_context.loc[original_text_anchor_long_context.project == project_name.upper(), "intermediate_risk"].values[0]
high_risk_anchor = original_text_anchor_long_context.loc[original_text_anchor_long_context.project == project_name.upper(), "high_risk"].values[0]

In [84]:
print(f"""
You are a clinical pathology writing assistant. Your task is to REWRITE the risk anchors so that each anchor (Low / Intermediate / High) **generalizes across** the provided patient reports while remaining clinically faithful.

### Inputs for {project_name.upper()}
1) Current anchor drafts:
- Low-risk (draft):
{low_risk_anchor}

- Intermediate-risk (draft):
{intermediate_risk_anchor}

- High-risk (draft):
{high_risk_anchor}

2) Patient reports to align with:
- Low-risk examples:
• {low_risk_descriptions[0]}
• {low_risk_descriptions[1]}
• {low_risk_descriptions[2]}

- Intermediate-risk examples:
• {intermediate_risk_descriptions[0]}
• {intermediate_risk_descriptions[1]}
• {intermediate_risk_descriptions[2]}

- High-risk examples:
• {high_risk_descriptions[0]}
• {high_risk_descriptions[1]}
• {high_risk_descriptions[2]}

### What to do
- Start from the current drafts but **refine** them to cover the common patterns across the example reports.
- Prioritize **risk-defining axes** over raw size: differentiation/grade, depth and extent of invasion (organ-wall layers, fat/sinus/peritoneum/adjacent organs), lymph node status (including NX), margin status (especially vascular/soft-tissue margins), and vascular/perineural invasion. Mention size qualitatively only if helpful (e.g., “small/moderate/large”), avoid exact measurements.
- If the disease/site can be inferred from the reports, use **site-appropriate anatomy terms** (e.g., pericolonic fat / renal sinus / perinephric fat). If uncertain, use neutral terms (“adjacent soft tissue”, “regional nodes”).
- **Do not copy** sentences verbatim from the reports. **Abstract and paraphrase**; prefer clinically neutral, guideline-like wording.
- Capture heterogeneity with inclusive phrasing (“confined to … or focal involvement of … without frank invasion of …”; “nodes negative when assessed; NX acceptable”).
- Keep each anchor concise: **5-10 sentences**, **≤ 500 words** per risk.
- **No JSON, no lists, no meta commentary.** Output plain prose only.

### Output format (plain text, exactly this order)
Low-Risk Anchor:
<one short paragraph>

Intermediate-Risk Anchor:
<one short paragraph>

High-Risk Anchor:
<one short paragraph>
""")



You are a clinical pathology writing assistant. Your task is to REWRITE the risk anchors so that each anchor (Low / Intermediate / High) **generalizes across** the provided patient reports while remaining clinically faithful.

### Inputs for STAD
1) Current anchor drafts:
- Low-risk (draft):
Gastric adenocarcinoma at this stage is generally limited in extent, often well to moderately differentiated, though high-grade histology may occasionally be present. Invasion is confined to the mucosa, submucosa, or subserosal adipose tissue without frank penetration of adjacent organs. Regional lymph nodes are negative when sampled, or lymph node status is not reported. Surgical margins are uninvolved, and distant metastasis is absent. Vascular or perineural invasion may be absent or focal, but overall disease remains localized and surgically controlled.

- Intermediate-risk (draft):
The gastric tumor shows moderately to poorly differentiated morphology with more extensive invasion into the musc